In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

# 1. Load and Clean Data
def load_and_preprocess(filepath):
    # Adjust column names as necessary based on your actual file
    df = pd.read_csv(filepath)
    df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df.set_index('Date', inplace=True)
    return df

# Replace with the path to your raw dataset
# df = load_and_preprocess('../data/brent_prices.csv')

# Placeholder data generation for execution check
dates = pd.date_range(start="1987-05-20", end="2022-09-30", freq="B")
np.random.seed(42)
prices = 20 + np.cumsum(np.random.normal(0, 1.5, len(dates)))
df = pd.DataFrame(data={"Price": np.clip(prices, 5, 150)}, index=dates)

# 2. Trend Analysis (Rolling Averages)
plt.figure(figsize=(14, 6))
plt.plot(df['Price'], label='Daily Price', color='blue', alpha=0.5)
plt.plot(df['Price'].rolling(window=252).mean(), label='1-Year Rolling Average (Trend)', color='red', linewidth=2)
plt.title('Brent Oil Price Trend Analysis')
plt.xlabel('Date')
plt.ylabel('USD per Barrel')
plt.legend()
plt.grid(True)
plt.show()

# 3. Calculate Log Returns
df['Log_Returns'] = np.log(df['Price']) - np.log(df['Price'].shift(1))
df.dropna(inplace=True)

# 4. Stationarity Testing (Augmented Dickey-Fuller Test)
def test_stationarity(timeseries, name):
    print(f"--- ADF Test Results for {name} ---")
    result = adfuller(timeseries)
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4E}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value:.4f}')
    if result[1] <= 0.05:
        print("Conclusion: Reject the null hypothesis (Series is Stationary).")
    else:
        print("Conclusion: Fail to reject the null hypothesis (Series is Non-Stationary).")
    print("-" * 40)

test_stationarity(df['Price'], 'Raw Prices')
test_stationarity(df['Log_Returns'], 'Log Returns')

# 5. Volatility Patterns (Log Returns Plot)
plt.figure(figsize=(14, 5))
plt.plot(df['Log_Returns'], color='purple', alpha=0.7)
plt.title('Brent Oil Log Returns (Volatility Clustering)')
plt.xlabel('Date')
plt.ylabel('Log Return')
plt.grid(True)
plt.show()